**Projet ISD2 final:** Analyse d'une base de données Formule 1! (2000-2024)

In [1]:
# importation des bibiliothéques nécéssaires au projet 
import pandas as pd 
import numpy as np
import os

In [2]:
# chargement des fichiers de source

Data_Dir = "DataSet_F1"                                 # pointe vers le fichier source du répositoire
Output = "DataSet_F1_Final"        
NA_VALUES = ["//N", ""] # nom du fichier complet

In [53]:
def load_csv(csv_file_name):
    file_path = os.path.join(Data_Dir, csv_file_name)
    return pd.read_csv(file_path, na_values = NA_VALUES)


In [54]:
def calculate_age(date_of_birth, current_year = 2026):
    year_of_birth = int(date_of_birth[:4])
    return current_year - year_of_birth

In [56]:
results           = load_csv("results.csv")
races             = load_csv("races.csv")
circuits          = load_csv("circuits.csv")
constructors      = load_csv("constructors.csv")
status            = load_csv("status.csv")
qualifying        = load_csv("qualifying.csv")
pit_stops         = load_csv("pit_stops.csv")
driver_standings  = load_csv("driver_standings.csv")
lap_times         = load_csv("lap_times.csv")
drivers           = load_csv("drivers.csv")

drivers["driver_name"] = drivers["forename"] + " " + drivers["surname"]
drivers["driver_age"] = drivers["dob"].apply(calculate_age)

pit_agg = pit_stops.groupby(["raceId", "driverId"]).agg(best_lap_ms = ("milliseconds", "min"), lap_std_ms = ("milliseconds", "std")).reset_index()
pits_aux  = pit_stops.groupby(["raceId", "driverId"]).count().reset_index()
pit_agg["pit_stop_count"] = pits_aux["stop"]

df = results.copy()
df = df.merge(races[["raceId", "year", "round", "circuitId", "date"]], on = "raceId", how = "left")
df = df.merge(circuits[["circuitId", "country"]], on = "circuitId", how = "left")
df = df.merge(drivers[["driverId", "driver_age" ]], on = "driverId", how = "left")
df = df.merge(constructors[["constructorId", "name"]].rename(columns = {"name" : "constructor_name"}), on = "constructorId", how = "left")
df = df.merge(status.rename(columns = {"status": "status_label"}), on = "statusId", how = "left")
df = df.merge(pit_agg, on = ["raceId", "driverId"], how = "left")

# on renomme chacune des colonnes(optionel mais pratique)

df = df.rename(columns = {
    "grid":         "grid_position",
    "positionOrder":  "finish_position",
    "points":       "points_scored",
    "laps":         "laps_completed",
    "status_label": "status",
    "country":      "circuit_country",
    "rank":         "fastest_lap_rank",
})

# choix de nos features -> 12 en total 

features = [
    "year",                                     
    "round", 
    "grid_position", 
    "finish_position",
    "points_scored",
    "laps_completed",
    "status",
    "constructor_name",
    "circuit_country",
    "driver_age",
    "pit_stop_count",
    "fastest_lap_rank",
]

df = df[features]
df = df[df["year"].between(2000, 2024)]        # reduit le nombre de lignes: passage de 1950-2024 à 2000-2024


df.to_csv(Output, index = False)

,year,round,grid_position,finish_position,points_scored,laps_completed,status,constructor_name,circuit_country,driver_age,pit_stop_count,fastest_lap_rank
0,2008,1,1,1,10.0,58,Finished,McLaren,Australia,41,NaN,2
1,2008,1,5,2,8.0,58,Finished,BMW Sauber,Australia,49,NaN,3
2,2008,1,7,3,6.0,58,Finished,Williams,Australia,41,NaN,5
3,2008,1,11,4,5.0,58,Finished,Renault,Australia,45,NaN,7
4,2008,1,3,5,4.0,58,Finished,McLaren,Australia,45,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...
26754,2024,24,14,16,0.0,57,+1 Lap,Haas F1 Team,UAE,34,4.0,1
26755,2024,24,12,17,0.0,55,Engine,RB F1 Team,UAE,24,3.0,12
26756,2024,24,9,18,0.0,30,Collision damage,Sauber,UAE,37,1.0,19
26757,2024,24,20,19,0.0,26,Engine,Williams,UAE,23,1.0,17
